# NOVA RETAIL — Reconciliación de SKUs

## Limpieza y homologación entre SAP y AS400
**Proyecto:** Diagnóstico de Prevención de Pérdidas  
**Autor:** Denz One  
**Fase:** Limpieza de datos y reconciliación de catálogos  
**Objetivo:** Construir una estrategia de homologación entre los catálogos SAP y AS400 para identificar equivalencias, detectar productos invisibles y preparar la base para la reconciliación de inventario.

---

### Propósito de este notebook
Este notebook tiene como objetivo resolver uno de los problemas más críticos del caso NOVA RETAIL:

- SAP y AS400 no comparten una tabla maestra de equivalencias
- un mismo producto puede existir con descripciones degradadas o formatos distintos
- ciertos productos de alto valor no existen en AS400
- sin reconciliar SKUs, cualquier análisis de merma o fraude queda contaminado

### Preguntas clave
1. ¿Qué tan comparable es el catálogo SAP con el catálogo AS400?
2. ¿Cuántos productos pueden reconciliarse automáticamente?
3. ¿Cuáles requieren revisión manual?
4. ¿Cuáles son invisibles para AS400 y representan riesgo estructural?

### Nota del analista
La reconciliación de SKUs no es un ejercicio cosmético de limpieza.  
Es una condición necesaria para poder estimar pérdida real, separar ruido operativo y detectar vulnerabilidades explotables en la cadena de inventario.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

DATA_PATH = Path("../data/raw")

products_sap = pd.read_csv(DATA_PATH / "products_sap.csv")
products_as400 = pd.read_csv(DATA_PATH / "products_as400.csv")

print("Datasets cargados correctamente.")
print("SAP:", products_sap.shape)
print("AS400:", products_as400.shape)

In [ ]:
products_sap.head()

In [ ]:
summary = pd.DataFrame({
    "dataset": ["products_sap", "products_as400"],
    "filas": [products_sap.shape[0], products_as400.shape[0]],
    "columnas": [products_sap.shape[1], products_as400.shape[1]]
})

summary

In [ ]:
apple_ghosts = products_as400[products_as400["is_ghost"] == True]

print(f"Total Apple Ghosts: {len(apple_ghosts)}")
print("\nPrimeros 5 Apple Ghosts:")
apple_ghosts.head()

In [ ]:
as400_quality = {
    "total_registros": len(products_as400),
    "sku_as400_nulos": products_as400["sku_as400"].isna().sum(),
    "description_as400_nulos": products_as400["description_as400"].isna().sum(),
    "category_as400_vacias": (products_as400["category_as400"] == "").sum(),
    "price_as400_nulos": products_as400["price_as400"].isna().sum(),
    "apple_ghosts": products_as400["is_ghost"].sum()
}

pd.DataFrame(as400_quality.items(), columns=["metrica", "valor"])

In [ ]:
corrupt_examples = products_as400[
    products_as400["description_as400"].notna() &
    products_as400["description_as400"].str.contains("SAMSNG|APLE|TV|PANT|REFRI|LICUAD|0|3", na=False)
].head(10)

corrupt_examples[["sku_as400", "description_as400", "price_as400"]]

In [ ]:
price_nulls_by_category = (
    products_as400[products_as400["price_as400"].isna()]
    ["category_as400"]
    .value_counts()
    .head(10)
)

price_nulls_by_category

In [ ]:
import plotly.express as px

quality_metrics = pd.DataFrame({
    "metrica": ["SKU nulos", "Descripciones nulas", "Categorías vacías", "Precios nulos", "Apple Ghosts"],
    "cantidad": [
        products_as400["sku_as400"].isna().sum(),
        products_as400["description_as400"].isna().sum(),
        (products_as400["category_as400"] == "").sum(),
        products_as400["price_as400"].isna().sum(),
        products_as400["is_ghost"].sum()
    ]
})

fig = px.bar(
    quality_metrics,
    x="metrica",
    y="cantidad",
    title="Problemas de calidad en catálogo AS400",
    color="cantidad",
    color_continuous_scale="Reds"
)

fig.show()

In [ ]:
import re

def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).upper().strip()

    replacements = {
        "SAMSNG": "SAMSUNG",
        "APLE": "APPLE",
        "TV": "TELEVISOR",
        "PANT": "PANTALLA",
        "REFRI": "REFRIGERADOR",
        "LICUAD": "LICUADORA",
        "0": "O",
        "3": "E",
        "_": " "
    }

    for old, new in replacements.items():
        text = text.replace(old, new)

    text = re.sub(r'[^A-Z0-9" ]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
products_sap["desc_norm"] = products_sap["description_sap"].apply(normalize_text)
products_as400["desc_norm"] = products_as400["description_as400"].apply(normalize_text)

products_sap[["description_sap", "desc_norm"]].head()

In [ ]:
products_as400[["description_as400", "desc_norm"]].head(15)

In [ ]:
import re

def normalize_text(text):
    if pd.isna(text):
        return None

    text = str(text).upper().strip()
    text = text.replace("_", " ")

    # Reemplazos seguros por palabra completa
    replacements = {
        r"\bSAMSNG\b": "SAMSUNG",
        r"\bAPLE\b": "APPLE",
        r"\bTV\b": "TELEVISOR",
        r"\bPANT\b": "PANTALLA",
        r"\bREFRI\b": "REFRIGERADOR",
        r"\bLICUAD\b": "LICUADORA",
    }

    for pattern, replacement in replacements.items():
        text = re.sub(pattern, replacement, text)

    # Correcciones controladas de caracteres
    text = text.replace("MONIT0R", "MONITOR")
    text = text.replace("T3LEVISOR", "TELEVISOR")
    text = text.replace("APPL3", "APPLE")

    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [ ]:
products_sap["desc_norm"] = products_sap["description_sap"].apply(normalize_text)
products_as400["desc_norm"] = products_as400["description_as400"].apply(normalize_text)

In [ ]:
products_as400[["description_as400", "desc_norm"]].head(15)

In [ ]:
top_sap_by_price = (
    products_sap
    .sort_values("price_sap", ascending=False)
    .head(20)
    [["sku_sap", "description_sap", "category_sap", "price_sap"]]
)

top_sap_by_price

In [ ]:
ghost_sap_refs = apple_ghosts["sku_sap_reference"].dropna().tolist()

sap_ghost_details = products_sap[
    products_sap["sku_sap"].isin(ghost_sap_refs)
][["sku_sap", "description_sap", "category_sap", "brand", "price_sap"]]

sap_ghost_details.sort_values("price_sap", ascending=False).head(20)

In [ ]:
import sys
sys.executable

In [ ]:
from fuzzywuzzy import process, fuzz
print("fuzzywuzzy cargado correctamente")

In [ ]:
sap_sample = products_sap.copy()
as400_sample = products_as400[products_as400["is_ghost"] == False].copy()

sap_sample = sap_sample.sort_values("price_sap", ascending=False).head(100)
as400_choices = as400_sample["desc_norm"].dropna().unique().tolist()

print("SAP sample:", sap_sample.shape)
print("AS400 choices:", len(as400_choices))

In [ ]:
sample_matches = []

for _, row in sap_sample.iterrows():
    desc = row["desc_norm"]

    if pd.isna(desc):
        continue

    best_match = process.extractOne(
        desc,
        as400_choices,
        scorer=fuzz.token_sort_ratio
    )

    if best_match:
        match_desc, score = best_match
        sample_matches.append({
            "sku_sap": row["sku_sap"],
            "description_sap": row["description_sap"],
            "price_sap": row["price_sap"],
            "matched_desc_as400": match_desc,
            "fuzzy_score": score
        })

sample_matches_df = pd.DataFrame(sample_matches)
sample_matches_df.sort_values("fuzzy_score", ascending=False).head(20)

In [ ]:
as400_lookup = products_as400[[
    "desc_norm", "price_as400", "sku_as400", "category_as400", "sku_sap_reference"
]].copy()

sample_matches_enriched = sample_matches_df.merge(
    as400_lookup,
    left_on="matched_desc_as400",
    right_on="desc_norm",
    how="left"
)

sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400",
    "fuzzy_score"
]].head(20)

In [ ]:
sample_matches_enriched["price_diff_abs"] = (
    sample_matches_enriched["price_sap"] - sample_matches_enriched["price_as400"]
).abs()

sample_matches_enriched["price_diff_pct"] = (
    sample_matches_enriched["price_diff_abs"] / sample_matches_enriched["price_sap"]
) * 100

sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400",
    "price_diff_pct",
    "fuzzy_score"
]].sort_values("price_diff_pct", ascending=False).head(20)

In [ ]:
def classify_match(row):
    if pd.isna(row["price_as400"]):
        return "AMBIGUO_PRECIO_NULO"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] <= 10:
        return "MATCH_CONFIABLE"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] > 10:
        return "AMBIGUO_PRECIO_INCONSISTENTE"
    else:
        return "MATCH_DEBIL"

sample_matches_enriched["match_status"] = sample_matches_enriched.apply(classify_match, axis=1)

sample_matches_enriched["match_status"].value_counts()

In [ ]:
sample_matches_enriched[[
    "sku_sap",
    "description_sap",
    "matched_desc_as400",
    "price_sap",
    "price_as400",
    "price_diff_pct",
    "fuzzy_score",
    "match_status"
]].sort_values(["match_status", "price_diff_pct"], ascending=[True, False]).head(30)

In [ ]:
products_as400["price_inferred_currency"] = np.where(
    products_as400["price_as400"].isna(),
    "UNKNOWN",
    np.where(products_as400["price_as400"] < 1000, "POTENTIAL_USD", "MXN")
)

products_as400["price_inferred_currency"].value_counts()

In [ ]:
products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency"
]].sample(20, random_state=42)

In [ ]:
high_value_categories = [
    "ELECTRONICA", "TELEFONIA", "COMPUTO", "ELECTRODOMESTICOS"
]

def infer_currency(row):
    price = row["price_as400"]
    category = row["category_as400"]

    if pd.isna(price):
        return "UNKNOWN"

    if category in high_value_categories or category in ["ELECTRON", "ELECTR", "PANTALLAS", "AUDIO Y VIDEO", "TEL", "CELULARES", "MOVILES"]:
        if price < 1000:
            return "POTENTIAL_USD"
        else:
            return "MXN"

    return "MXN"

products_as400["price_inferred_currency_v2"] = products_as400.apply(infer_currency, axis=1)

products_as400["price_inferred_currency_v2"].value_counts()

In [ ]:
products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency_v2"
]].sample(25, random_state=7)

In [ ]:
EXCHANGE_RATE = 17.3

products_as400["price_as400_adjusted"] = np.where(
    products_as400["price_inferred_currency_v2"] == "POTENTIAL_USD",
    products_as400["price_as400"] * EXCHANGE_RATE,
    products_as400["price_as400"]
)

products_as400[[
    "description_as400",
    "category_as400",
    "price_as400",
    "price_inferred_currency_v2",
    "price_as400_adjusted"
]].sample(20, random_state=11)

In [ ]:
as400_lookup_v2 = products_as400[[
    "desc_norm",
    "price_as400_adjusted",
    "sku_as400",
    "category_as400",
    "sku_sap_reference",
    "price_inferred_currency_v2"
]].copy()

sample_matches_enriched_v2 = sample_matches_df.merge(
    as400_lookup_v2,
    left_on="matched_desc_as400",
    right_on="desc_norm",
    how="left"
)

sample_matches_enriched_v2["price_diff_abs"] = (
    sample_matches_enriched_v2["price_sap"] - sample_matches_enriched_v2["price_as400_adjusted"]
).abs()

sample_matches_enriched_v2["price_diff_pct"] = (
    sample_matches_enriched_v2["price_diff_abs"] / sample_matches_enriched_v2["price_sap"]
) * 100

sample_matches_enriched_v2[[
    "sku_sap",
    "description_sap",
    "price_sap",
    "matched_desc_as400",
    "price_as400_adjusted",
    "price_diff_pct",
    "price_inferred_currency_v2",
    "fuzzy_score"
]].sort_values("price_diff_pct").head(25)

In [ ]:
def classify_match_v2(row):
    if pd.isna(row["price_as400_adjusted"]):
        return "AMBIGUO_PRECIO_NULO"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] <= 10:
        return "MATCH_CONFIABLE"
    elif row["fuzzy_score"] >= 90 and row["price_diff_pct"] > 10:
        return "AMBIGUO_PRECIO"
    else:
        return "MATCH_DEBIL"

sample_matches_enriched_v2["match_status_v2"] = sample_matches_enriched_v2.apply(classify_match_v2, axis=1)

sample_matches_enriched_v2["match_status_v2"].value_counts()

In [ ]:
sample_matches_enriched_v2[[
    "sku_sap",
    "description_sap",
    "matched_desc_as400",
    "price_sap",
    "price_as400_adjusted",
    "price_diff_pct",
    "fuzzy_score",
    "match_status_v2"
]].sort_values(["match_status_v2", "price_diff_pct"], ascending=[True, True]).head(30)

In [ ]:
sample_matches_enriched_v2["category_sap"] = sample_matches_enriched_v2["sku_sap"].map(
    products_sap.set_index("sku_sap")["category_sap"]
)

ambiguos = sample_matches_enriched_v2[
    sample_matches_enriched_v2["match_status_v2"] == "AMBIGUO_PRECIO"
]

ambiguos["category_sap"].value_counts()

In [ ]:
ambiguos[[
    "sku_sap",
    "description_sap",
    "category_sap",
    "price_sap",
    "price_as400_adjusted",
    "price_diff_pct",
    "fuzzy_score"
]].sort_values("price_diff_pct", ascending=False).head(25)

In [ ]:
products_sap["priority_flag"] = np.where(
    products_sap["price_sap"] >= products_sap["price_sap"].quantile(0.95),
    "TOP_5_VALUE",
    "STANDARD"
)

priority_skus = products_sap[products_sap["priority_flag"] == "TOP_5_VALUE"].copy()

print("SKUs prioritarios:", len(priority_skus))
priority_skus[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(20)

In [ ]:
products_sap["priority_flag"] = np.where(
    products_sap["price_sap"] >= products_sap["price_sap"].quantile(0.95),
    "TOP_5_VALUE",
    "STANDARD"
)

priority_skus = products_sap[products_sap["priority_flag"] == "TOP_5_VALUE"].copy()

print("SKUs prioritarios:", len(priority_skus))
priority_skus[["sku_sap", "description_sap", "category_sap", "price_sap"]].head(20)

In [ ]:
resumen_reconciliacion = pd.DataFrame({
    "indicador": [
        "SKUs SAP totales",
        "SKUs AS400 totales",
        "Apple Ghosts",
        "SKUs prioritarios (Top 5% valor)",
        "Matches piloto evaluados",
        "Matches confiables (piloto)",
        "Matches ambiguos (piloto)",
        "Precios AS400 nulos"
    ],
    "valor": [
        len(products_sap),
        len(products_as400),
        int(products_as400["is_ghost"].sum()),
        int((products_sap["priority_flag"] == "TOP_5_VALUE").sum()),
        len(sample_matches_enriched_v2),
        int((sample_matches_enriched_v2["match_status_v2"] == "MATCH_CONFIABLE").sum()),
        int((sample_matches_enriched_v2["match_status_v2"] == "AMBIGUO_PRECIO").sum()),
        int(products_as400["price_as400"].isna().sum())
    ]
})

resumen_reconciliacion

## Conclusiones de la fase de reconciliación

### Hallazgos principales
- El catálogo AS400 presenta problemas estructurales de trazabilidad, especialmente por categorías vacías, precios nulos y productos invisibles.
- Los **Apple Ghosts** representan un riesgo crítico porque existen en SAP pero no en AS400, impidiendo su seguimiento completo en el entorno legacy.
- La normalización textual permitió construir una primera capa funcional de matching entre catálogos.
- El matching difuso por descripción, sin validación financiera, genera equivalencias semánticamente plausibles pero financieramente inconsistentes.
- La inferencia de moneda mejoró materialmente la comparabilidad de precios en productos de alto valor.
- La reconciliación total del catálogo no es operativamente eficiente en esta fase; el enfoque correcto es una **reconciliación selectiva por valor**.

### Decisión metodológica
La siguiente etapa del proyecto debe priorizar los SKUs de mayor valor económico, en lugar de intentar una limpieza horizontal de los 14,000 productos. Esta decisión maximiza la recuperación potencial y reduce el tiempo de diagnóstico.

### Riesgo residual
Persisten ambigüedades en una parte relevante del catálogo legacy, especialmente en productos de alto ticket con discrepancias de valuación. Estos casos requieren tratamiento controlado antes de escalar a reconciliación masiva.

## Próximo paso recomendado

Con la reconciliación inicial de SKUs establecida, la siguiente fase del análisis debe enfocarse en:

1. **Cruzar despachos del CEDIS contra recepciones en tienda**
2. **Normalizar fechas y unidades de recepción**
3. **Medir varianzas selectivas por tienda, categoría y horario**
4. **Identificar tiendas con patrones incompatibles con error operativo aleatorio**

El objetivo ya no es solo limpiar datos, sino comenzar a separar:
- ruido administrativo
- falla operativa
- y pérdida potencialmente intencional
